# Mumbai Data Preprocessing

Combine EXIF metadata (video info + GPS timeseries) with annotation data to create a merged dataset.

In [ ]:
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

In [ ]:
BASE_DIR = Path('../..')

VIDEO_METADATA_FILES = [
    BASE_DIR / '1_8_exif_video_metadata.csv',
    BASE_DIR / '9_11_exif_video_metadata.csv',
    BASE_DIR / '13_19_6_7_exif_video_metadata.csv',
]

GPS_TIMESERIES_FILES = [
    BASE_DIR / '1_8_gps_timeseries.csv',
    BASE_DIR / '9_11_gps_timeseries.csv',
    BASE_DIR / '13_19_6_7_gps_timeseries_new.csv',
]

ANNOTATION_FILE = BASE_DIR / 'labelstudio/mumbai_export_221408_project-221408-at-2026-02-02-21-33-2247d6c5.json'

FRAME_METADATA_FILES = [
    BASE_DIR / '1_8_frame_metadata.log',
    BASE_DIR / '9_11_frame_metadata.log',
    BASE_DIR / '13_19_6_7_frame_metadata.log',
]

OUTPUT_FILE = BASE_DIR / 'data/mumbai_annotations_with_exif.csv'

## 1. Load Video Metadata

In [ ]:
video_dfs = []
for f in VIDEO_METADATA_FILES:
    if f.exists():
        df = pd.read_csv(f)
        video_dfs.append(df)
        print(f"{f.name}: {len(df)} videos")

video_metadata = pd.concat(video_dfs, ignore_index=True)
print(f"\nTotal videos: {len(video_metadata)}")
print(f"Columns: {list(video_metadata.columns)}")

In [ ]:
video_metadata[['video_id', 'recording_datetime', 'video_duration_sec', 'camera_model']].head(10)

## 2. Load GPS Timeseries

In [ ]:
def dms_to_decimal(dms_str):
    """Convert DMS format to decimal degrees."""
    if pd.isna(dms_str):
        return None
    match = re.match(r"(\d+) deg (\d+)' ([\d.]+)\" ([NSEW])", str(dms_str))
    if match:
        d, m, s, direction = match.groups()
        decimal = float(d) + float(m)/60 + float(s)/3600
        if direction in ['S', 'W']:
            decimal = -decimal
        return decimal
    return None

def parse_altitude(alt_str):
    """Parse altitude string like '11.815 m' to float."""
    if pd.isna(alt_str):
        return None
    match = re.match(r"([\d.-]+)\s*m", str(alt_str))
    if match:
        return float(match.group(1))
    return None

In [ ]:
gps_dfs = []
for f in GPS_TIMESERIES_FILES:
    if f.exists():
        df = pd.read_csv(f)
        gps_dfs.append(df)
        print(f"{f.name}: {len(df):,} GPS points")

gps_timeseries = pd.concat(gps_dfs, ignore_index=True)
print(f"\nTotal GPS points: {len(gps_timeseries):,}")

In [ ]:
gps_timeseries['lat'] = gps_timeseries['gps_latitude'].apply(dms_to_decimal)
gps_timeseries['lon'] = gps_timeseries['gps_longitude'].apply(dms_to_decimal)
gps_timeseries['alt'] = gps_timeseries['gps_altitude'].apply(parse_altitude)

gps_timeseries['gps_datetime'] = pd.to_datetime(gps_timeseries['gps_datetime'], format='%Y:%m:%d %H:%M:%S.%f', errors='coerce')

print(f"GPS points with valid lat/lon: {gps_timeseries['lat'].notna().sum():,}")
gps_timeseries[['video_id', 'gps_datetime', 'lat', 'lon', 'alt']].head()

## 3. Load Frame Metadata

In [ ]:
def parse_frame_metadata_log(filepath):
    """Parse pipe-delimited frame metadata log file."""
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('|')
            if len(parts) >= 10:
                rows.append({
                    'base_video_id': parts[0],
                    'source_folder': parts[1],
                    'video_name': parts[2],
                    'original_filename': parts[3],
                    'frame_filename': parts[4],
                    'frame_number_raw': int(parts[5]),
                    'timestamp_sec': float(parts[6]),
                    'timestamp_str': parts[7],
                    'fps': float(parts[8]),
                    'video_duration_sec': float(parts[9])
                })
    return pd.DataFrame(rows)

frame_dfs = []
for f in FRAME_METADATA_FILES:
    if f.exists():
        df = parse_frame_metadata_log(f)
        frame_dfs.append(df)
        print(f"{f.name}: {len(df):,} frames")

frame_metadata = pd.concat(frame_dfs, ignore_index=True)
print(f"\nTotal frames: {len(frame_metadata):,}")

In [ ]:
frame_metadata.head()

## 4. Load Annotations

In [ ]:
with open(ANNOTATION_FILE, 'r') as f:
    annotation_data = json.load(f)

print(f"Total annotation tasks: {len(annotation_data)}")

In [ ]:
def parse_annotation(task, annotation):
    """Parse a single annotation into a flat dict."""
    row = {
        'task_id': task['id'],
        'annotation_id': annotation['id'],
        'annotator_email': annotation['completed_by']['email'],
        'image': task['data']['image'],
        'file_upload': task.get('file_upload', ''),
    }
    
    for result in annotation['result']:
        field_name = result['from_name']
        if result['type'] == 'taxonomy':
            value = result['value']['taxonomy'][0][0] if result['value']['taxonomy'] else None
        elif result['type'] == 'choices':
            value = result['value']['choices'][0] if result['value']['choices'] else None
        elif result['type'] == 'textarea':
            value = result['value']['text'][0] if result['value']['text'] else None
        else:
            value = None
        row[field_name] = value
    
    return row

rows = []
for task in annotation_data:
    for annotation in task['annotations']:
        rows.append(parse_annotation(task, annotation))

annotations = pd.DataFrame(rows)
print(f"Total annotations: {len(annotations)}")

In [ ]:
core_fields = ['men_count', 'women_count', 'men_twowheeler', 'women_twowheeler', 
               'footpath', 'lane_markings', 'potholes', 'litter']
skip_fields = ['bus_station', 'railway_station', 'street_vendor']

def is_skip_row(row):
    has_core = any(pd.notna(row.get(f)) for f in core_fields if f in row.index)
    has_skip = all(
        f in row.index and pd.notna(row.get(f)) and row.get(f) == 'No' 
        for f in skip_fields
    )
    return (not has_core) and has_skip

skip_mask = annotations.apply(is_skip_row, axis=1)
print(f"Skip rows: {skip_mask.sum()}")
print(f"Valid annotations: {(~skip_mask).sum()}")

annotations = annotations[~skip_mask].copy()

## 5. Extract Frame Info from Image Paths

In [ ]:
def extract_frame_info(image_path):
    """Extract video_id, frame_number, and timestamp from image path.
    
    Example: upload/221408/c594e56e-3_itinerary_7_frame05100_t000250_170.jpg
    """
    filename = image_path.split('/')[-1]
    
    match = re.match(r'[a-f0-9]+-(.+)_frame(\d+)_t(\d+)_(\d+)\.jpg', filename)
    if match:
        base_video_id = match.group(1)
        frame_number = int(match.group(2))
        timestamp_parts = match.group(3)
        frame_idx = int(match.group(4))
        
        if len(timestamp_parts) == 6:
            minutes = int(timestamp_parts[:3])
            seconds = int(timestamp_parts[3:])
            timestamp_sec = minutes * 60 + seconds
        else:
            timestamp_sec = float(timestamp_parts)
            
        return {
            'base_video_id': base_video_id,
            'frame_number': frame_number,
            'timestamp_sec': timestamp_sec,
            'frame_idx': frame_idx
        }
    return None

frame_info = annotations['image'].apply(extract_frame_info).apply(pd.Series)
annotations = pd.concat([annotations, frame_info], axis=1)
print(f"Extracted frame info for {annotations['base_video_id'].notna().sum()} annotations")

In [ ]:
annotations[['image', 'base_video_id', 'frame_number', 'timestamp_sec']].head(10)

## 6. Match Annotations to Video Metadata

In [ ]:
def extract_video_base_id(video_id):
    """Extract base video ID without hash suffix.
    
    Example: 1_itinerary_1_1_8e5f0fc4 -> 1_itinerary_1_1
    """
    parts = video_id.rsplit('_', 1)
    if len(parts) == 2 and len(parts[1]) == 8:
        return parts[0]
    return video_id

video_metadata['base_video_id'] = video_metadata['video_id'].apply(extract_video_base_id)

print("Sample video_id mappings:")
print(video_metadata[['video_id', 'base_video_id']].head(10))

In [ ]:
annotations_with_video = annotations.merge(
    video_metadata[['video_id', 'base_video_id', 'source_folder', 'recording_datetime', 
                    'video_duration_sec', 'camera_model', 'video_fps']],
    on='base_video_id',
    how='left'
)

print(f"Annotations matched to video: {annotations_with_video['video_id'].notna().sum()}")
print(f"Annotations without video match: {annotations_with_video['video_id'].isna().sum()}")

In [ ]:
if annotations_with_video['video_id'].isna().any():
    unmatched = annotations_with_video[annotations_with_video['video_id'].isna()]['base_video_id'].unique()
    print(f"Unmatched base_video_ids ({len(unmatched)}):")
    for vid in unmatched[:20]:
        print(f"  {vid}")

## 7. Interpolate GPS Coordinates for Each Frame

In [ ]:
video_metadata['recording_datetime_parsed'] = pd.to_datetime(
    video_metadata['recording_datetime'], 
    format='%Y:%m:%d %H:%M:%S', 
    errors='coerce'
)

video_start_times = video_metadata.set_index('video_id')['recording_datetime_parsed'].to_dict()

In [ ]:
def get_gps_for_frame(row, gps_df, video_start_times):
    """Get GPS coordinates for a frame by interpolating from GPS timeseries."""
    video_id = row.get('video_id')
    timestamp_sec = row.get('timestamp_sec')
    
    if pd.isna(video_id) or pd.isna(timestamp_sec):
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None})
    
    video_gps = gps_df[gps_df['video_id'] == video_id].copy()
    if len(video_gps) == 0:
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None})
    
    video_start = video_start_times.get(video_id)
    if pd.isna(video_start):
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None})
    
    frame_datetime = video_start + timedelta(seconds=timestamp_sec)
    
    video_gps['time_diff'] = abs((video_gps['gps_datetime'] - frame_datetime).dt.total_seconds())
    closest = video_gps.loc[video_gps['time_diff'].idxmin()]
    
    if closest['time_diff'] > 30:
        return pd.Series({'gps_lat': None, 'gps_lon': None, 'gps_alt': None})
    
    return pd.Series({
        'gps_lat': closest['lat'],
        'gps_lon': closest['lon'],
        'gps_alt': closest['alt']
    })

In [ ]:
print("Matching GPS coordinates to frames (this may take a moment)...")

gps_coords = annotations_with_video.apply(
    lambda row: get_gps_for_frame(row, gps_timeseries, video_start_times),
    axis=1
)

annotations_final = pd.concat([annotations_with_video, gps_coords], axis=1)

print(f"Annotations with GPS: {annotations_final['gps_lat'].notna().sum()}")
print(f"Annotations without GPS: {annotations_final['gps_lat'].isna().sum()}")

## 8. Convert Count Fields to Numeric

In [ ]:
count_cols = ['men_count', 'women_count', 'men_twowheeler', 'women_twowheeler']

def convert_count(val):
    if pd.isna(val):
        return np.nan
    if val == '>10':
        return 11
    try:
        return int(val)
    except (ValueError, TypeError):
        return np.nan

for col in count_cols:
    if col in annotations_final.columns:
        annotations_final[col] = annotations_final[col].apply(convert_count)

## 9. Save Merged Dataset

In [ ]:
output_columns = [
    'task_id', 'annotation_id', 'annotator_email', 'image',
    'base_video_id', 'video_id', 'frame_number', 'timestamp_sec',
    'source_folder', 'recording_datetime', 'video_duration_sec', 'camera_model',
    'gps_lat', 'gps_lon', 'gps_alt',
    'men_count', 'women_count', 'men_twowheeler', 'women_twowheeler',
    'footpath', 'lane_markings', 'potholes', 'litter',
    'bus_station', 'railway_station', 'street_vendor'
]

available_cols = [c for c in output_columns if c in annotations_final.columns]
output_df = annotations_final[available_cols].copy()

print(f"Output columns: {len(available_cols)}")
print(f"Output rows: {len(output_df)}")

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
output_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved to {OUTPUT_FILE}")

## 10. Summary

In [ ]:
print("=" * 50)
print("PREPROCESSING SUMMARY")
print("=" * 50)
print(f"\nInput:")
print(f"  Videos in metadata: {len(video_metadata)}")
print(f"  GPS points: {len(gps_timeseries):,}")
print(f"  Annotations: {len(annotations)}")

print(f"\nOutput:")
print(f"  Total rows: {len(output_df)}")
print(f"  With video metadata: {output_df['video_id'].notna().sum()}")
print(f"  With GPS coordinates: {output_df['gps_lat'].notna().sum()}")

print(f"\nGPS Coverage:")
if output_df['gps_lat'].notna().any():
    print(f"  Lat range: {output_df['gps_lat'].min():.4f} to {output_df['gps_lat'].max():.4f}")
    print(f"  Lon range: {output_df['gps_lon'].min():.4f} to {output_df['gps_lon'].max():.4f}")

In [ ]:
output_df.head(10)